# View match between predicted probabilities and participant choices

Need to load the participant data from the csv in the form of trials and then look at the match between the simulated performance the participant actual performance

## 1. Loading participant data

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Union

class ParticipantDataLoader:
    """
    A class to load and manage participant data from the wine_quality.csv file.
    Allows easy access to trials for specific participants and phases.
    """
    
    def __init__(self, csv_path: str):
        """
        Initialize the data loader with the CSV file.
        
        Args:
            csv_path (str): Path to the wine_quality.csv file
        """
        self.data = pd.read_csv(csv_path)
        self.participants = self._get_participant_info()
        
    def _get_participant_info(self) -> Dict:
        """Get unique participant information."""
        participant_info = {}
        for participant_id in self.data['Participant Id'].unique():
            participant_data = self.data[self.data['Participant Id'] == participant_id].iloc[0]
            participant_info[participant_id] = {
                'condition': participant_data['Condition'],
                'model': participant_data['Model'],
                'app_id': participant_data['AppId'],
                'complexity': participant_data['Complexity']
            }
        return participant_info
    
    def get_participant_ids(self) -> List:
        """Get list of all participant IDs."""
        return list(self.participants.keys())
    
    def get_participant_info(self, participant_id) -> Dict:
        """Get general info for a specific participant."""
        return self.participants.get(participant_id, {})
    
    def get_participant_trials(self, participant_id, phase: Optional[str] = None) -> pd.DataFrame:
        """
        Get all trials for a specific participant.
        
        Args:
            participant_id: The participant ID
            phase (str, optional): Filter by phase ('forward' or 'counterfactual')
            
        Returns:
            pd.DataFrame: Filtered trial data
        """
        participant_data = self.data[self.data['Participant Id'] == participant_id]
        
        if phase:
            participant_data = participant_data[participant_data['Phase'] == phase]
            
        return participant_data.sort_values('Trial Index')
    
    def get_forward_trials(self, participant_id) -> pd.DataFrame:
        """
        Get forward phase trials for a participant with relevant columns.
        
        Args:
            participant_id: The participant ID
            
        Returns:
            pd.DataFrame: Forward trial data with relevant columns
        """
        forward_data = self.get_participant_trials(participant_id, 'forward')
        
        forward_columns = [
            'Participant Id', 'Trial Index', 'Instance Id', 'XAIType', 'Tested w/ XAI', 'Time',
            'Response', 'AI prediction', 'DT prediction', 'LR prediction', 
            'Explainer prediction', 'Response==AI', 'Response==DT', 
            'Response==LR', 'Response==Explainer'
        ]
        
        # Only include columns that exist in the data
        available_columns = [col for col in forward_columns if col in forward_data.columns]
        
        return forward_data[available_columns]
    
    def get_counterfactual_trials(self, participant_id) -> pd.DataFrame:
        """
        Get counterfactual phase trials for a participant with relevant columns.
        
        Args:
            participant_id: The participant ID
            
        Returns:
            pd.DataFrame: Counterfactual trial data with relevant columns
        """
        cf_data = self.get_participant_trials(participant_id, 'counterfactual')
        
        cf_columns = [
            'Participant Id', 'Trial Index', 'Instance Id', 'XAIType', 'Tested w/ XAI', 'Time',
            'Changed feature index', 'Changed feature name', 'Changed feature type',
            'Changed from', 'Changed to', 'Changed amount', 'DT prediction (CF)',
            'LR prediction (CF)', 'Changed prediction (DT)', 'Changed prediction (LR)',
            'Changed explainer'
        ]
        
        # Only include columns that exist in the data
        available_columns = [col for col in cf_columns if col in cf_data.columns]
        
        return cf_data[available_columns]
    
    def get_trial_by_index(self, participant_id, trial_index: int) -> pd.DataFrame:
        """
        Get a specific trial by trial index for a participant.
        
        Args:
            participant_id: The participant ID
            trial_index (int): The trial index number
            
        Returns:
            pd.DataFrame: Single trial data
        """
        participant_data = self.get_participant_trials(participant_id)
        return participant_data[participant_data['Trial Index'] == trial_index]
    
    def get_xai_performance(self, participant_id, xai_type: str = None) -> pd.DataFrame:
        """
        Get trials filtered by XAI type and whether XAI was shown.
        
        Args:
            participant_id: The participant ID
            xai_type (str, optional): Filter by specific XAI type
            
        Returns:
            pd.DataFrame: Filtered trial data
        """
        participant_data = self.get_participant_trials(participant_id)
        
        if xai_type:
            participant_data = participant_data[participant_data['XAIType'] == xai_type]
            
        return participant_data
    
    def summarize_participant_performance(self, participant_id) -> Dict:
        """
        Get a summary of participant performance across both phases.
        
        Args:
            participant_id: The participant ID
            
        Returns:
            Dict: Summary statistics
        """
        forward_trials = self.get_forward_trials(participant_id)
        cf_trials = self.get_counterfactual_trials(participant_id)
        
        summary = {
            'participant_info': self.get_participant_info(participant_id),
            'total_trials': len(self.get_participant_trials(participant_id)),
            'forward_trials': len(forward_trials),
            'counterfactual_trials': len(cf_trials),
            'avg_time_forward': forward_trials['Time'].mean() if not forward_trials.empty else 0,
            'avg_time_counterfactual': cf_trials['Time'].mean() if not cf_trials.empty else 0,
        }
        
        # Add accuracy metrics for forward trials if available
        if not forward_trials.empty:
            accuracy_columns = ['Response==AI', 'Response==DT', 'Response==LR', 'Response==Explainer']
            for col in accuracy_columns:
                if col in forward_trials.columns:
                    summary[f'accuracy_{col.split("==")[1].lower()}'] = forward_trials[col].mean()
        
        return summary
    
    def get_instance_data(self, instance_id: int) -> pd.DataFrame:
        """
        Get all trials for a specific instance across all participants.
        
        Args:
            instance_id (int): The instance ID
            
        Returns:
            pd.DataFrame: All trials for the instance
        """
        return self.data[self.data['Instance Id'] == instance_id]
    
    def compare_participants(self, participant_ids: List) -> pd.DataFrame:
        """
        Compare performance metrics across multiple participants.
        
        Args:
            participant_ids (List): List of participant IDs to compare
            
        Returns:
            pd.DataFrame: Comparison metrics
        """
        comparison_data = []
        
        for pid in participant_ids:
            summary = self.summarize_participant_performance(pid)
            comparison_data.append({
                'Participant Id': pid,
                **summary['participant_info'],
                'Total Trials': summary['total_trials'],
                'Forward Trials': summary['forward_trials'],
                'Counterfactual Trials': summary['counterfactual_trials'],
                'Avg Time Forward': summary['avg_time_forward'],
                'Avg Time Counterfactual': summary['avg_time_counterfactual'],
                **{k: v for k, v in summary.items() if k.startswith('accuracy_')}
            })
        
        return pd.DataFrame(comparison_data)

# Example usage:
# loader = ParticipantDataLoader('datasets/wine_quality.csv')
# participant_ids = loader.get_participant_ids()
# print(f"Found {len(participant_ids)} participants")
# 
# # Get data for first participant
# first_participant = participant_ids[0]
# forward_trials = loader.get_forward_trials(first_participant)
# cf_trials = loader.get_counterfactual_trials(first_participant)
# summary = loader.summarize_participant_performance(first_participant)
# 
# print(f"Participant {first_participant} summary:")
# print(summary)

In [ ]:

# Example usage:
loader = ParticipantDataLoader('datasets/wine_quality.csv')
participant_ids = loader.get_participant_ids()
forward_trials = loader.get_forward_trials(participant_ids[0])
cf_trials = loader.get_counterfactual_trials(participant_ids[0])
summary = loader.summarize_participant_performance(participant_ids[0])

In [ ]:
participant_ids

## 2. Load the AI dataset loader, explainers and forward simulation methods

In [ ]:
from src.utils import AIDatasetLoader, filter_by_app_and_model, DecisionTreeInterpreter, LogisticRegressionInterpreter  
from src.memory import Chunk, DeclarativeMemory

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
current_dir = os.getcwd()
data_dir = os.path.join(current_dir, 'datasets')

file_values = os.path.join(data_dir, 'values.csv')
file_metadata = os.path.join(data_dir, 'metadata.csv')
file_prediction = os.path.join(data_dir, 'none.csv')

values_df = pd.read_csv(file_values)
metadata_df = pd.read_csv(file_metadata)
prediction_df = pd.read_csv(file_prediction)

# ✅ Which model to use per dataset
dataset_model_map = {
    "mushrooms": "mlp",
    "wine_quality": "mlp",
    "forest_cover": "xgboost",
    "adult": "xgboost",
}

# # 🔧 Choose dataset here:
# app_id = "wine_quality"  # 🔄 Change this line to switch datasets

# # 🧠 Auto-configured values:
# model_name = dataset_model_map[app_id]

In [ ]:
# load ai loader
ai_dataset_loader = AIDatasetLoader(
    feature_values_df=values_df,
    metadata_df=metadata_df,
    AI_predictions_df=prediction_df
)

In [ ]:
# Load decision tree
dt_df = pd.read_csv(os.path.join(data_dir, 'decision_tree.csv'))
# dt_exp = DecisionTreeInterpreter(dt_df, metadata_df, app_id, model_name, depth=2)
# dt_exp.print_tree(as_name=True)


# Load linear model
lr_df = pd.read_csv(os.path.join(data_dir, 'logistic_regression.csv'))
# lr_exp = LogisticRegressionInterpreter(lr_df, metadata_df, app_id, model_name, variant="sparse")

In [ ]:
lr_exp = LogisticRegressionInterpreter(lr_df, metadata_df, "wine_quality", "mlp", variant="sparse")
lr_exp.coefficients

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import random

# import importlib
# import src.memory as memory
# importlib.reload(memory)
# import src.dt_memory as dt_memory
# importlib.reload(dt_memory)
# import src.heuristic_lr_model as heuristic_lr_model
# importlib.reload(heuristic_lr_model)
# import src.lr_memory as lr_memory
# importlib.reload(lr_memory)

# from src.memory import DeclarativeMemory, CombinedMemory
# from src.dt_memory import (
#     add_node_to_memory, predict_with_dt_memory_prob, read_dt_prob,
#     remember_with_dt_feedback, get_feature_nums
# )
# from src.heuristic_lr_model import (
#     add_heuristic_lr_prob_to_memory, predict_with_heuristic_lr_prob_memory,
#     remember_with_heuristic_lr_prob_feedback
# )
# from src.lr_memory import (
#     add_lr_to_memory, predict_with_lr_memory_prob, read_lr_prob, remember_with_lr_feedback
# )

# # Interpreters + helpers (assumed)
# # from src.interpreters import DecisionTreeInterpreter, LogisticRegressionInterpreter
# # from src.ai_dataset_loader import filter_by_app_and_model

# DEFAULT_DM_PARAMS = dict(
#     retrieval_threshold=0.0,
#     latency_factor=0.3,
#     latency_exponent=0.2,
#     max_assoc_strength=2.0,
#     mismatch_penalty=-3.0,
#     activation_noise=1.0
# )

# def run_participant_sim(
#     loader,
#     ai_dataset_loader,
#     dt_df,
#     lr_df,
#     metadata_df,
#     participant_index: int,
#     T_READ_NUM: float = 2.0,
#     W0_ANS: float = 0.1,
#     dm_params: dict = None,
#     use_lr_heuristic: bool = True
# ):
#     """
#     Run the trial simulation for a specific participant (by index in loader.get_participant_ids()).
#     Returns (df, summary) where df has one row per trial and summary has quick metrics.
#     """
#     dm_params = dm_params or DEFAULT_DM_PARAMS

#     # 1) Pick participant by index
#     participant_ids = loader.get_participant_ids()
#     if participant_index < 0 or participant_index >= len(participant_ids):
#         raise IndexError(f"participant_index out of range (0..{len(participant_ids)-1})")
#     # shortlist the participants whose condition is 'lr'
#     participant_ids = [pid for pid in participant_ids if loader.get_participant_info(pid)['condition'] == 'LR']

#     if not participant_ids:
#         raise ValueError("No participants found with condition 'LR'")

#     participant_id = participant_ids[participant_index]
#     info = loader.get_participant_info(participant_id)

#     print(info)

#     app_id     = info['app_id']
#     model_name = info['model']
#     condition  = info['condition']
#     complexity = info['complexity']

#     # 2) Build explainer by condition/complexity
#     if condition == 'DT':
#         depth = 3 if complexity == 'high' else 2
#         explainer  = DecisionTreeInterpreter(dt_df, metadata_df, app_id, model_name, depth=depth)
#         model_type = "dt"
#     elif condition == 'LR':
#         variant   = "sparse" if complexity == 'low' else "dense"
#         explainer = LogisticRegressionInterpreter(lr_df, metadata_df, app_id, model_name, variant=variant)
#         model_type = "lr"
#     else:
#         raise ValueError(f"Unknown condition: {condition}")

#     # 3) Narrow dataset loader to app/model
#     ai_loader = filter_by_app_and_model(ai_dataset_loader, app_id, model_name)

#     # 4) Memory setup
#     dm = DeclarativeMemory(**dm_params)
#     memory = CombinedMemory(dm, wm_capacity=10)

#     # 5) (Optional) seed memory with explainer knowledge
#     if model_type == "dt":
#         add_node_to_memory(explainer.tree_structure[0], memory, dt_exp=explainer, feature_nums=get_feature_nums(explainer))
#     # elif model_type == "lr":
#     #     add_lr_to_memory(explainer, memory)    # if you want the exact-LR-chunk route

#     memory.tick(90)

#     # 6) Iterate trials
#     forward_trials = loader.get_forward_trials(participant_id)
#     records = []
#     first_instance = True

#     for _, row in forward_trials.iterrows():
#         instance_id     = row['Instance Id']
#         with_xai        = row['Tested w/ XAI']          # "w/ XAI" or "w/o XAI"
#         trial_number    = int(row['Trial Index'])
#         actual_response = float(row["Response"])
#         response_time   = float(row["Time"])

#         instances, preds = ai_loader.load_instances([instance_id], normalize=(model_type.lower()!="dt"))
#         instance, _pred  = instances[0], preds[0]

#         unnorm_instance = ai_loader.load_instances([instance_id], normalize=False)[0][0]

#         # Predict + (optionally) remember feedback
#         if with_xai == "w/ XAI":
#             if model_type == "lr":
#                 if use_lr_heuristic:
#                     if first_instance:
#                         add_heuristic_lr_prob_to_memory(explainer, memory, initial_instance=instance, initial_sigma=3.0)
#                         first_instance = False
#                     prob_vec, pred_time, info = predict_with_heuristic_lr_prob_memory(
#                         instance, memory, lr_exp=explainer, T_READ_NUM=T_READ_NUM, verbose=True,
#                         active_indices=[0, 1, 3],
#                         W0_ANS=W0_ANS
#                     )
#                     if info.get("mean_z", None) is None:
#                         print(info)
#                     remember_with_heuristic_lr_prob_feedback(
#                         memory, explainer, instance,
#                         actual=(explainer.apply_to_instance(unnorm_instance) > 0),
#                         predicted=int(np.argmax(prob_vec)),
#                         mean_z=info.get("mean_z", None),
#                         active_indices=[0, 1, 3],
#                         verbose=True
#                     )
#                 else:
#                     # prob_vec, pred_time, info = read_dt_prob(instance, explainer, T_READ_NUM=T_READ_NUM, verbose=False)
#                     # remember_with_lr_feedback(memory, explainer)
#                     pass
#             else:
#                 prob_vec, pred_time, info = read_dt_prob(explainer, instance, T_READ_NUM=T_READ_NUM)
#                 remember_with_dt_feedback(instance, memory, feature_nums=get_feature_nums(explainer), verbose=True)
#         else:
#             if model_type == "lr":
#                 if use_lr_heuristic:
#                     prob_vec, pred_time, _ = predict_with_heuristic_lr_prob_memory(
#                         instance, memory, lr_exp=explainer, T_READ_NUM=T_READ_NUM, verbose=True,
#                         active_indices=[0, 1, 3],
#                         W0_ANS=W0_ANS
#                     )
#                     remember_with_heuristic_lr_prob_feedback(
#                         memory, explainer, instance,
#                         actual=(explainer.apply_to_instance(unnorm_instance) > 0),
#                         predicted=int(np.argmax(prob_vec)),
#                         mean_z=info.get("mean_z", None),
#                         active_indices=[0, 1, 3],
#                         verbose=True
#                     )
#                 else:
#                     pass
#             else:
#                 prob_vec, pred_time, _ = predict_with_dt_memory_prob(instance, memory, dt_exp=explainer, T_READ_NUM=T_READ_NUM)

#         # Probability mass assigned to participant's actual response
#         prob_of_resp = prob_vec[0] if actual_response < 0 else prob_vec[1]

#         # Explainer's own class prediction
#         if model_type == 'lr':
#             expl_pred = int(bool(explainer.apply_to_instance(unnorm_instance) > 0))
#         else:
#             expl_pred = int(explainer.apply_to_instance(instance)["class_index"])

#         match_expl = int(int(actual_response > 0) == expl_pred)


#         print(f"Trial Index: {trial_number}")
#         print(f"Instance Id: {instance_id}, x: {instance}")
#         print(f"Tested w/ XAI: {with_xai}, Actual response: {actual_response}, Probability of response: {prob_of_resp}, Match: {match_expl}")
#         if prob_of_resp < 0.5:
#             print("Model disagrees with participant")
#         print()
#         records.append(dict(
#             trial=trial_number,
#             instance_id=instance_id,
#             with_xai=with_xai,
#             actual_response=actual_response,
#             prob_of_response=float(prob_of_resp),
#             pred_time=float(pred_time),
#             resp_time=float(response_time),
#             residual_time=float(response_time - pred_time),
#             explainer_pred=expl_pred,
#             model_match_explainer=int(np.argmax(prob_vec)==expl_pred),
#             participant_match_explainer=match_expl
#         ))

#     df = pd.DataFrame.from_records(records).sort_values("trial").reset_index(drop=True)
#     summary = dict(
#         participant_id=participant_id,
#         app_id=app_id, model_name=model_name,
#         condition=condition, complexity=complexity,
#         avg_match=float(np.mean(df["participant_match_explainer"])) if len(df) else np.nan,
#         avg_model_match_explainer=float(np.mean(df["model_match_explainer"])) if len(df) else np.nan,
#         avg_model_match_participant=float(np.mean(df["prob_of_response"]>0.5)) if len(df) else np.nan
#     )
#     return df, summary


In [ ]:

# def plot_participant_results(df: pd.DataFrame):
#     """Make the 4 plots given the df produced by run_participant_sim()."""
#     # 1) Time series: predicted vs actual
#     plt.figure()
#     plt.plot(df["trial"], df["resp_time"], marker="o", label="Actual RT")
#     plt.plot(df["trial"], df["pred_time"], marker="s", label="Predicted RT")
#     plt.xlabel("Trial"); plt.ylabel("Time (s)")
#     plt.title("Response Time per Trial"); plt.legend(); plt.tight_layout(); plt.show()

#     # 2) Scatter: predicted vs actual RT (with y=x)
#     plt.figure()
#     plt.scatter(df["pred_time"], df["resp_time"], marker="o")
#     dmin = float(min(df["pred_time"].min(), df["resp_time"].min()))
#     dmax = float(max(df["pred_time"].max(), df["resp_time"].max()))
#     plt.plot([dmin, dmax], [dmin, dmax])
#     plt.xlabel("Predicted Time (s)"); plt.ylabel("Actual Time (s)")
#     plt.title("Predicted vs Actual Response Time"); plt.tight_layout(); plt.show()

#     # 3) Probability of participant’s chosen response
#     plt.figure()
#     plt.plot(df["trial"], df["prob_of_response"], marker="o")
#     plt.ylim(0.0, 1.0)
#     plt.xlabel("Trial"); plt.ylabel("P(model assigns to participant's response)")
#     plt.title("Predicted Probability of Participant's Chosen Response")

#     # add horizontal line at 0.5
#     plt.axhline(0.5, color="gray", linestyle="--", linewidth=1)

#     # compute metrics
#     avg_prob = df["prob_of_response"].mean()
#     avg_nll = -(np.log(df["prob_of_response"]).mean())

#     # annotate under graph
#     textstr = f"Avg Prob = {avg_prob:.3f}\nAvg NLL = {avg_nll:.3f}"
#     plt.gcf().text(0.5, -0.05, textstr, ha="center", va="top")

#     plt.tight_layout(); plt.show()

#     # compute metrics
#     avg_prob = df["prob_of_response"].mean()
#     avg_nll = -(np.log(df["prob_of_response"]).mean())

#     # annotate under graph
#     textstr = f"Avg Prob = {avg_prob:.3f}\nAvg NLL = {avg_nll:.3f}"
#     plt.gcf().text(0.5, -0.05, textstr, ha="center", va="top")

#     plt.tight_layout(); plt.show()

#     # 4) Split by XAI condition (w/ XAI vs w/o XAI), mild outlier trim
#     if "with_xai" in df.columns:
#         for label in ["w/ XAI", "w/o XAI"]:
#             sub = df[df["with_xai"] == label]
#             if len(sub) == 0: 
#                 continue
#             sub = sub[sub["resp_time"] < sub["resp_time"].quantile(0.95)]
#             if len(sub) == 0:
#                 continue
#             plt.figure()
#             plt.plot(sub["trial"], sub["resp_time"], marker="o", label="Actual RT")
#             plt.plot(sub["trial"], sub["pred_time"], marker="s", label="Predicted RT")
#             plt.xlabel("Trial"); plt.ylabel("Time (s)")
#             plt.title(f"Response Time per Trial • {label}")
#             plt.legend(); plt.tight_layout(); plt.show()

In [ ]:


# def run_random_participant():
#     participant_index = 15#np.random.randint(1, 30)  # choose any valid index
    

#     df, summary = run_participant_sim(
#         loader, ai_dataset_loader, dt_df, lr_df, metadata_df,
#         participant_index=participant_index,
#         T_READ_NUM=2.0,

#         W0_ANS=0.0,
#         dm_params=DEFAULT_DM_PARAMS,
#         use_lr_heuristic=True
#     )

#     if summary['avg_match'] < 0.6:
#         print(f"Participant {participant_index} is shit.")
#         run_random_participant()
#     else:
#         print("Participant Index: ", participant_index)
#         print("Avg participant match with explainer:", summary['avg_match'])
#         print("Avg model match with explainer:", summary['avg_model_match_explainer'])
#         print("Avg model match with participant:", summary['avg_model_match_participant'])

#         # print the two columns instance_id, model_match_explainer, participant_match_explainer
#         print(df[["instance_id", "model_match_explainer", "participant_match_explainer"]])

#         plot_participant_results(df)


# run_random_participant()

## 3. Fitting parameters to participant data

### re-imports

In [ ]:

import importlib
import src.memory as memory
importlib.reload(memory)
import src.dt_memory as dt_memory
importlib.reload(dt_memory)
import src.heuristic_lr_model as heuristic_lr_model
importlib.reload(heuristic_lr_model)
import src.lr_memory as lr_memory
importlib.reload(lr_memory)

from src.memory import DeclarativeMemory, CombinedMemory
from src.dt_memory import (
    add_dt_to_memory, dt_traverse, refresh_dt_path_in_memory
)
from src.heuristic_lr_model import (
    add_lr_heuristic_to_memory, lr_heuristic, refresh_lr_heuristic_in_memory
)
from src.lr_memory import (
    add_lr_calculation_to_memory, lr_calculation, refresh_lr_calculation_in_memory
)

from typing import Optional
import random
from dataclasses import replace

In [ ]:
# ====== GP-based BO for LR-only model (concise + consistent) ======

import math, numpy as np
from dataclasses import dataclass
from typing import Optional

# ---------------- Utils ----------------
def bernoulli_nll(p, eps=1e-4):
    p = max(min(float(p), 1.0 - eps), eps)
    return -math.log(p)

def _clamp(x, lo, hi): return lo if x < lo else (hi if x > hi else x)

def _snap_compute_sf(x_cont):
    # map [1,3] -> {1,2,3}
    return 1 if x_cont < 1.5 else (2 if x_cont < 2.5 else 3)

# Robust MSE for time with outlier trimming (default: drop top 5% of participant times)
def _mae_time_qtrim(rt_true, rt_pred, q=0.95):
    if not rt_true:
        return 0.0
    t = np.asarray(rt_true, float)
    p = np.asarray(rt_pred, float)
    thresh = np.quantile(t, q)
    mask = t <= thresh
    if not np.any(mask):
        return 0.0
    print(f"Average Participant Timing: {np.mean(t[mask]):.3f}s (n={np.sum(mask)})")
    print(f"Average Model Timing: {np.mean(p[mask]):.3f}s")    

    return float(np.mean(np.abs(t[mask] - p[mask])))

# Parameter grouping
LOG_PARAMS = ["T_enc", "T_op", "latency_factor", "lapse"]           # >0, multiplicative scale
LIN_PARAMS = ["retrieval_threshold", "ddm_a", "ddm_s", "compute_sf"] # signed/small/categorical

# Order in optimizer vector (log_ prefix for log-params; compute_sf uses a continuous proxy)
FULL_ORDER = [
    "log_T_enc", "log_T_op", "retrieval_threshold",
    "log_latency_factor", "ddm_a", "ddm_s", "log_lapse", "compute_sf_cont"
]

# ---------------- Memory builder ----------------
def _make_memory(retrieval_threshold, latency_factor):
    dm = DeclarativeMemory(
        retrieval_threshold=retrieval_threshold,
        latency_factor=latency_factor,
        latency_exponent=1.0,
        max_assoc_strength=2.0,
        mismatch_penalty=-1.5,
        activation_noise=0.3,
        decay=0.5,
    )
    return CombinedMemory(dm, wm_capacity=7)

def _select_prob(probs, y):
    if hasattr(probs, "__len__") and len(probs) == 2:
        return float(probs[1]) if float(y) > 0 else float(probs[0])
    raise ValueError(f"Unexpected probs format: {probs}")


# ---------------- Strategy adapters ----------------
def _seed_memory_for_strategy(strategy, memory, explainer, compute_sf):
    """
    Initializes memory for the chosen strategy.
    """
    if strategy == "lr_calc":
        # preload LR coefficients for calculation path
        add_lr_calculation_to_memory(explainer, memory)
    elif strategy == "lr_heur":
        # preload LR heuristic state (use your preferred init var)
        add_lr_heuristic_to_memory(explainer, memory, initial_var=0.1)
    elif strategy == "dt":
        # preload DT structure (threshold sig figs follow compute_sf)
        add_dt_to_memory(memory, explainer, thresh_sf=int(compute_sf))
    else:
        raise ValueError(f"Unknown strategy: {strategy!r}")


def _predict_once(strategy, instance, norm_instance, memory, explainer, *, with_xai,
                  T_enc, T_op, ddm_a, ddm_s, compute_sf):
    """
    Runs ONE trial under the chosen strategy and returns:
      probs, pred_time, aux_info
    """
    if strategy == "lr_calc":
        # lr_calculation in 'read' or 'retrieve'
        probs, pred_time, aux = lr_calculation(
            instance, memory, lr_exp=explainer,
            T_enc=T_enc, T_op=T_op, ddm_a=ddm_a, ddm_s=ddm_s,
            compute_sf=compute_sf,
            mode=("read" if with_xai else "retrieve"),
        )
        return probs, pred_time, aux

    elif strategy == "lr_heur":
        # heuristic maps T_enc->T_READ_NUM, T_op->T_INTUITIVE_OP
        probs, pred_time, info = lr_heuristic(
            norm_instance, memory, explainer,
            num_samples=40, K_top=3,
            T_enc=T_enc,
            ddm_a=ddm_a, ddm_s=ddm_s, ddm_Tnd=0.30, ddm_norm="l2",
            active_indices=None, verbose=False
        )
        return probs, pred_time, info

    elif strategy == "dt":
        # dt_traverse in 'read' or 'retrieve'
        probs, pred_time, aux = dt_traverse(
            instance, memory, explainer,
            mode=("read" if with_xai else "retrieve"),
            compute_sf=int(compute_sf),
            T_enc=T_enc, ddm_a=ddm_a, ddm_s=ddm_s, ddm_Tnd=0.30, ddm_norm="l2",
            n_mc=64, topk_k=3, refresh_prob_cap=1.0, verbose=False
        )
        return probs, pred_time, aux

    else:
        raise ValueError(f"Unknown strategy: {strategy!r}")


def _post_read_refresh(strategy, memory, explainer, *, compute_sf, info_or_aux, instance, with_xai, actual_label):
    """
    Applies post-read/online refresh depending on strategy.
    - For lr_calc: refresh only when with_xai=True (as before).
    - For lr_heur: refresh on BOTH. If with_xai=False, use model’s own pred as 'actual'.
    - For dt:      after read, refresh the path; retrieve-mode may use refresh_prob_cap internally.
    """
    if strategy == "lr_calc":
        if with_xai:
            refresh_lr_calculation_in_memory(
                memory, explainer,
                intercept_display_sf=int(compute_sf),
                factor_display_sf=int(compute_sf),
            )

    elif strategy == "lr_heur":
        # 'info_or_aux' is the 'info' dict returned by lr_heuristic
        info = info_or_aux
        # actual_label already set by caller to either human response (with XAI) or model pred (w/o XAI)
        refresh_lr_heuristic_in_memory(
            memory, explainer, info, actual=int(actual_label),
            active_indices=None, w_min=1e-4, verbose=False
        )

    elif strategy == "dt":
        if with_xai:
            # refresh the path after reading explanations
            # (thresh_sf ties to compute_sf for consistency)
            refresh_dt_path_in_memory(memory, explainer, instance, thresh_sf=int(compute_sf))


# ---------------- Trial simulation (updated) ----------------
def _simulate_trials(
    T_enc, T_op, ddm_a, ddm_s, retrieval_threshold, latency_factor,
    forward_trials, ai_dataset_loader, explainer, model_name, app_id: str,
    compute_sf=2, lapse=0.0, *, strategy: str = "lr_calc"
):
    memory = _make_memory(retrieval_threshold, latency_factor)
    local_loader = filter_by_app_and_model(ai_dataset_loader, app_id, model_name)

    _seed_memory_for_strategy(strategy, memory, explainer, compute_sf)
    memory.tick(90)

    prob_list, rt_pred, rt_true = [], [], []
    trials = forward_trials.sort_values("Trial Index")

    for _, row in trials.iterrows():
        instance_id   = int(row["Instance Id"])
        actual_resp   = int(row["Response"])
        response_time = float(row["Time"])
        with_xai      = str(row.get("Tested w/ XAI", "w/o XAI")).strip().lower() in ("w/ xai","with xai","xai","1","true","yes")

        instances, preds = local_loader.load_instances([instance_id], normalize=False)
        instance = instances[0]

        norm_instance = local_loader.load_instances([instance_id], normalize=True)[0][0]

        # 1) forward pass
        probs, pred_time, aux = _predict_once(
            strategy, instance, norm_instance, memory, explainer,
            with_xai=with_xai,
            T_enc=T_enc, T_op=T_op, ddm_a=ddm_a, ddm_s=ddm_s,
            compute_sf=compute_sf
        )

        # 2) NLL with post-lapse mixing
        p = _select_prob(probs, actual_resp)
        if lapse > 0:
            p = (1.0 - lapse) * p + 0.5 * lapse

        # 3) Online update / refresh (strategy-specific)
        if strategy == "lr_heur":
            # If no XAI, use model’s own prediction as "actual"
            if not with_xai:
                label_for_update = preds[0]
            else:
                label_for_update = int(explainer.apply_to_instance(instance)>0)
            _post_read_refresh(
                "lr_heur", memory, explainer,
                compute_sf=compute_sf, info_or_aux=aux,
                instance=instance, with_xai=with_xai, actual_label=label_for_update
            )
        elif strategy == "lr_calc":
            _post_read_refresh(
                "lr_calc", memory, explainer,
                compute_sf=compute_sf, info_or_aux=aux,
                instance=instance, with_xai=with_xai, actual_label=None
            )
        elif strategy == "dt":
            _post_read_refresh(
                "dt", memory, explainer,
                compute_sf=compute_sf, info_or_aux=aux,
                instance=instance, with_xai=with_xai, actual_label=None
            )

        # 4) record metrics
        prob_list.append(float(p))
        rt_pred.append(float(pred_time))
        rt_true.append(response_time)

    return prob_list, rt_pred, rt_true

# ---------------- Objective ----------------
@dataclass
class ObjConfig:
    w_resp: float = 10.0
    w_time: float = 0.0
    repeats: int = 1

    # bounds named exactly as parameters
    T_enc_bounds: tuple = (0.05, 10.0)
    T_op_bounds: tuple = (0.05, 1.0)
    retrieval_threshold_bounds: tuple = (-2.0, -0.1)
    latency_factor_bounds: tuple = (0.3, 3.0)
    ddm_a_bounds: tuple = (0.5, 3.0)
    ddm_s_bounds: tuple = (0.5, 2.0)
    lapse_bounds: tuple = (0.0, 0.3)
    compute_sf_bounds: tuple = (1.0, 3.0)  # continuous proxy domain

def make_objective(forward_trials, ai_dataset_loader, explainer, model_name, cfg: ObjConfig, app_id: str, strategy: str):
    def obj_components(params: dict):
        total_resp, total_time = 0.0, 0.0
        nrep = max(1, int(cfg.repeats))

        for _ in range(nrep):
            probs, rt_pred, rt_true = _simulate_trials(
                params["T_enc"], params["T_op"], params["ddm_a"], params["ddm_s"],
                params["retrieval_threshold"], params["latency_factor"],
                forward_trials, ai_dataset_loader, explainer, model_name, app_id,
                compute_sf=params["compute_sf"], lapse=params["lapse"], strategy=strategy
            )

            nll_resp = sum(bernoulli_nll(p) for p in probs)

            # >>> Outlier-trimmed MAE for timing (drop top 10% of participant times)
            mae_time = _mae_time_qtrim(rt_true, rt_pred, q=0.80)

            total_resp += nll_resp
            total_time += mae_time

        nll_resp /= nrep
        mae_time /= nrep
        return {
            "total": cfg.w_resp * nll_resp + cfg.w_time * mae_time,
            "respNLL": nll_resp,
            "timeMAE": mae_time
        }

    # Vector<->params adapters (short, generic)
    def obj_from_x(x):
        vals = dict(zip(FULL_ORDER, x))
        # log -> exp and clamp to bounds
        out = {}
        for name in LOG_PARAMS:
            lo, hi = getattr(cfg, f"{name}_bounds")
            v = math.exp(vals[f"log_{name}"])
            out[name] = _clamp(v, lo, hi)
        # linear ones
        for name in LIN_PARAMS:
            if name == "compute_sf":
                continue
            lo, hi = getattr(cfg, f"{name}_bounds")
            out[name] = _clamp(float(vals[name]), lo, hi)
        # compute_sf via snapping
        lo, hi = cfg.compute_sf_bounds
        out["compute_sf"] = _snap_compute_sf(_clamp(float(vals["compute_sf_cont"]), lo, hi))

        comps = obj_components(out)
        comps.update(out)
        return comps

    return obj_components, obj_from_x

# ---------------- GP-BO ----------------
def fit_params_gp_bo(
    forward_trials, ai_dataset_loader, explainer, model_name, app_id=None,
    init_vals=None, n_calls=60, n_initial_points=12, random_state=0,
    cfg: ObjConfig=ObjConfig(), verbose=True, freeze: Optional[dict]=None,
    strategy: str = None
):
    from skopt import gp_minimize
    from skopt.space import Real

    obj_components, obj_from_x = make_objective(
        forward_trials, ai_dataset_loader, explainer, model_name, cfg, app_id, strategy
    )

    freeze = (freeze or {}).copy()
    init_vals = init_vals or {
        "T_enc": 2.0, "T_op": 0.2, "retrieval_threshold": -3.0,
        "latency_factor": 0.3, "ddm_a": 1.0, "ddm_s": 1.0,
        "lapse": 0.1, "compute_sf": 2.0,
    }

    # bounds map in optimizer coords (auto from cfg)
    bounds = {
        "log_T_enc": tuple(map(math.log, cfg.T_enc_bounds)),
        "log_T_op": tuple(map(math.log, cfg.T_op_bounds)),
        "retrieval_threshold": cfg.retrieval_threshold_bounds,
        "log_latency_factor": tuple(map(math.log, cfg.latency_factor_bounds)),
        "ddm_a": cfg.ddm_a_bounds,
        "ddm_s": cfg.ddm_s_bounds,
        "log_lapse": tuple(map(lambda z: math.log(z + 1e-5), cfg.lapse_bounds)),
        "compute_sf_cont": cfg.compute_sf_bounds,
    }

    # initial x for any name (generic)
    def _x0(name):
        if name.startswith("log_"):
            base = name[4:]
            return math.log(freeze.get(base, init_vals[base]) + (1e-5 if base == "lapse" else 0.0))
        if name == "compute_sf_cont":
            return float(freeze.get("compute_sf", init_vals["compute_sf"]))
        return float(freeze.get(name, init_vals[name]))

    # build skopt space, skipping frozen (check base names)
    space, unfrozen = [], []
    for name in FULL_ORDER:
        base = name[4:] if name.startswith("log_") else ("compute_sf" if name == "compute_sf_cont" else name)
        if base not in freeze:
            lo, hi = bounds[name]
            space.append(Real(lo, hi, name=name))
            unfrozen.append(name)

    # expander for a reduced x vector
    def _expand(x_unfrozen):
        full = {n: _x0(n) for n in FULL_ORDER}
        for n, v in zip(unfrozen, x_unfrozen): full[n] = v
        return tuple(full[n] for n in FULL_ORDER)

    eval_log = []

    def f_obj(x_unfrozen):
        comps = obj_from_x(_expand(x_unfrozen))
        eval_log.append(comps)
        if verbose:
            print(
                f"[BO {len(eval_log):03d}] y={comps['total']:.3f} | "
                f"respNLL={comps['respNLL']:.3f}, timeMAE={comps['timeMAE']:.3f} | "
                f"T_enc={comps['T_enc']:.3f}, T_op={comps['T_op']:.3f} | "
                f"thr={comps['retrieval_threshold']:.3f}, L={comps['latency_factor']:.3f}, "
                f"a={comps['ddm_a']:.3f}, s={comps['ddm_s']:.3f}, lapse={comps['lapse']:.3f}, "
                f"csf={comps['compute_sf']}"
            )
        return comps["total"]

    # all frozen? just evaluate once
    if not space:
        best = obj_from_x(_expand([]))
        best.update({"objective": best["total"], "optimizer": "GP(skopt) (all frozen)", "n_evals": 1, "history": [best]})
        return best

    x0 = [_x0(n) for n in unfrozen]
    res0 = obj_from_x(_expand(x0))

    result = gp_minimize(
        f_obj, space, x0=[x0], y0=[res0["total"]],
        n_calls=n_calls, n_initial_points=n_initial_points,
        acq_func="EI", random_state=random_state, noise=1e-6
    )

    best = obj_from_x(_expand(result.x))
    best.update({
        "objective": result.fun,
        "optimizer": "GP(skopt)",
        "n_evals": len(eval_log),
        "history": eval_log,
    })
    return best


In [ ]:
# Get participant info
participant_ids = loader.get_participant_ids()
# Pick random participant with LR + high complexity
participant_id = random.choice([
    pid for pid in participant_ids
    if ((loader.get_participant_info(pid)['condition'] == 'LR') &
        (loader.get_participant_info(pid)['complexity'] == 'high'))
])
# participant_id = "REDACTED_PARTICIPANT_ID"

print(f"Optimizing for participant {participant_id}...")

participant_info = loader.get_participant_info(participant_id)

app_id     = participant_info['app_id']
model_name = participant_info['model']
condition  = participant_info['condition']
complexity = participant_info['complexity']

# Set up LR explainer (dense for high, sparse for low)
if condition == 'LR':
    variant = "sparse" if complexity == 'low' else "dense"
    explainer = LogisticRegressionInterpreter(
        lr_df, metadata_df, app_id, model_name, variant=variant
    )
else:
    explainer = DecisionTreeInterpreter(
        dt_df, metadata_df, app_id, model_name, depth=3 if complexity == 'high' else 2
    )


# Dataset loader restricted to participant’s app/model
ai_dataset_loader1 = filter_by_app_and_model(ai_dataset_loader, app_id, model_name)
forward_trials = loader.get_forward_trials(participant_id)


strategy = "lr_heur"  # "lr_calc", "lr_heur", "dt"

# @dataclass
# class ObjConfig:
#     w_resp: float = 10.0
#     w_time: float = 0.0
#     repeats: int = 1

#     # bounds named exactly as parameters
#     T_enc_bounds: tuple = (0.05, 10.0)
#     T_op_bounds: tuple = (0.05, 1.0)
#     retrieval_threshold_bounds: tuple = (-2.0, -0.1)
#     latency_factor_bounds: tuple = (0.3, 3.0)
#     ddm_a_bounds: tuple = (0.5, 3.0)
#     ddm_s_bounds: tuple = (0.5, 2.0)
#     lapse_bounds: tuple = (0.0, 0.3)
#     compute_sf_bounds: tuple = (1.0, 3.0)  # continuous proxy domain


# Config
cfg = ObjConfig(
    w_resp=1.0, w_time=0.5, repeats=1,
    T_enc_bounds=(0.5, 3.0),
    T_op_bounds=(2.0, 3.0),
    retrieval_threshold_bounds=(-2.0, 1.5),
    latency_factor_bounds=(0.001, 1.0),
    ddm_a_bounds=(0.8, 1.2),
    ddm_s_bounds=(0.8, 1.2),
    lapse_bounds=(0.05, 0.2),
    compute_sf_bounds=(1.0, 3.0)
)

init_vals = {
        "T_enc": 3.0, "T_op": 2.0, "retrieval_threshold": 0.0,
        "latency_factor": 1.0, "ddm_a": 1.0, "ddm_s": 0.8,
        "lapse": 0.1, "compute_sf": 2.0,
}

# Run BO with canonical freeze keys
res = fit_params_gp_bo(
    forward_trials, ai_dataset_loader1, explainer, model_name, app_id=app_id, strategy=strategy,
    init_vals=init_vals,
    n_calls=50, n_initial_points=1, random_state=42,
    cfg=cfg, verbose=True,
    freeze={
        "T_op": 2.0,            # freeze T_op to 2.0
        "lapse": 0.05,          # freeze lapse
        "compute_sf": 2,        # freeze compute_sf
        # "T_enc": 1.5,           # freeze T_enc
        "latency_factor": 1.0,
        # "ddm_a": 1.0,
        # "ddm_s": 1.0
        # "retrieval_threshold": 0.2
    }
)

print(
    "\nBest params:\n" + "\n".join(
        f"{k}: {v:.3f}" if isinstance(v, (int, float)) else f"{k}: {v}"
        for k, v in res.items() if k != "history"
    )
)




### functions and their parameter

###### HEURISTIC LR
add_lr_heuristic_to_memory(lr_exp, memory, initial_var: float = 0.25)
def lr_heuristic(
    feature_vector,
    memory,
    lr_exp,
    *,
    num_samples: int = 40,          # dont need to vary
    K_top: int = 3,                 # top-k to consider per retrieval, dont need to vary
    T_READ_NUM: float = 2.0,        # per numeric/categorical read
    T_INTUITIVE_OP: float = 0.5,    # per internal +/*, dont need to vary
    # DDM params (tune a, s, Tnd; norm as in your helper)
    ddm_a: float = 1.5,
    ddm_s: float = 1.0,
    ddm_Tnd: float = 0.30, # no need to vary
    ddm_norm: str = "l2",
    active_indices: list = None, # no need to vary
    verbose: bool = False,
): returns     info = {
        "decision": {
            "p1": p1,
            "v_ratio_mean": float(np.mean(v_s)),
        },
        "timing": {
            "retrieval_rt_sum": float(retrieval_cost),
            "read_time_sum": float(read_cost),
            "ddm_rt_mean": float(np.mean(rt_s)),
            "total_time": total_time,
        },
        "chunks": {
            "intercept": {
                "chosen_name": (r_int["top_k"][0][0].name if r_int["top_k"] else None),
            },
            "features": [
                {
                    "key": key,
                    "value": float(val),
                    "chosen_name": (r_coef["top_k"][0][0].name if r_coef["top_k"] else None),
                    "is_numeric": bool(is_numeric),
                }
                for (key, val, is_numeric, r_coef) in feat_meta
            ]
        }
    }

    return probs, total_time, info

def refresh_lr_heuristic_in_memory(
    memory,
    lr_exp,
    info,                 # ← from predict()
    actual: int,
    *,
    active_indices: list[int] = None,  # optional feature filter
    w_min: float = 1e-4,                      # min curvature p(1-p) for stability
    verbose: bool = False,
):

###### DT    
def add_dt_to_memory(memory, dt_exp, *, thresh_sf: int = 2)
def dt_traverse(
    feature_vector,
    memory, dt_exp,
    *,
    mode: str = "retrieve",          # "retrieve" or "read"
    compute_sf: int = 2,
    T_enc: float = 2.0,
    # DDM params
    ddm_a: float = 1.5,
    ddm_s: float = 1.0,
    ddm_Tnd: float = 0.30,
    ddm_norm: str = "l2",
    # MC / retrieval planning
    n_mc: int = 64,
    topk_k: int = 3,
    refresh_prob_cap: float = 1.0,   # cap per post-hoc refresh (retrieve-mode only)
    verbose: bool = False,
):
def refresh_dt_path_in_memory(
    memory, dt_exp, feature_vector, *, thresh_sf: int = 2
):


In [ ]:
res.keys()

In [ ]:
# import numpy as np
# from src.lr_memory import ddm_prob_rt_ratio

# def demo_table(a_list=(0.5, 1.0, 1.5, 2.0), s_list=(0.5, 1.0, 1.5), c=1.0, xs=None):
#     if xs is None:
#         xs = np.linspace(-3, 3, 13)  # coarse sweep
    
#     print(f"{'a':>3} {'s':>3} {'x':>6} {'sum':>8} {'den(L2)':>10} {'v_ratio':>9} {'p_up':>8} {'E_T':>8}")
#     for a in a_list:
#         for s in s_list:
#             for x in xs:
#                 terms = [float(x), -float(c)]
#                 p_up, E_T, v_ratio, denom = ddm_prob_rt_ratio(
#                     terms, a=a, s=s, Tnd=0.30, norm="l2"
#                 )
#                 print(f"{a:>3.1f} {s:>3.1f} {x:>6.2f} {sum(terms):>8.3f} {denom:>10.3f} {v_ratio:>9.3f} {p_up:>8.3f} {E_T:>8.3f}")

# # run it
# demo_table()

# import numpy as np
# import matplotlib.pyplot as plt

# def curve(a=1.5, s=1.0, c=1.0, xs=None, Tnd=0.30, norm="l2"):
#     if xs is None:
#         xs = np.linspace(-3, 3, 301)
#     p_list, t_list, v_list = [], [], []
#     for x in xs:
#         terms = [float(x), -float(c)]
#         p_up, E_T, v_ratio, _ = ddm_prob_rt_ratio(terms, a=a, s=s, Tnd=Tnd, norm=norm)
#         p_list.append(p_up); t_list.append(E_T); v_list.append(v_ratio)
#     return xs, np.array(p_list), np.array(t_list), np.array(v_list)

# # --- Plot 1: vary 'a' at fixed 's'
# s_fixed = 1.0
# a_vals = [0.5, 1.0, 1.5, 2.0]
# plt.figure()
# for a in a_vals:
#     xs, p, t, v = curve(a=a, s=s_fixed, c=1.0)
#     plt.plot(xs, p, label=f"a={a}, s={s_fixed}")
# plt.axvline(1.0, ls="--", alpha=0.5)   # where sum crosses zero (x=c)
# plt.title("p_up vs x (varying a, s fixed)")
# plt.xlabel("x (with terms=[x, -c], c=1)")
# plt.ylabel("p_up")
# plt.legend()
# plt.tight_layout()
# plt.show()

# plt.figure()
# for a in a_vals:
#     xs, p, t, v = curve(a=a, s=s_fixed, c=1.0)
#     plt.plot(xs, t, label=f"a={a}, s={s_fixed}")
# plt.axvline(1.0, ls="--", alpha=0.5)
# plt.title("E[T] vs x (varying a, s fixed)")
# plt.xlabel("x (with terms=[x, -c], c=1)")
# plt.ylabel("E[T] (s)")
# plt.legend()
# plt.tight_layout()
# plt.show()

# # --- Plot 2: vary 's' at fixed 'a'
# a_fixed = 1.5
# s_vals = [0.5, 1.0, 1.5]
# plt.figure()
# for s_ in s_vals:
#     xs, p, t, v = curve(a=a_fixed, s=s_, c=1.0)
#     plt.plot(xs, p, label=f"a={a_fixed}, s={s_}")
# plt.axvline(1.0, ls="--", alpha=0.5)
# plt.title("p_up vs x (varying s, a fixed)")
# plt.xlabel("x (with terms=[x, -c], c=1)")
# plt.ylabel("p_up")
# plt.legend()
# plt.tight_layout()
# plt.show()

# plt.figure()
# for s_ in s_vals:
#     xs, p, t, v = curve(a=a_fixed, s=s_, c=1.0)
#     plt.plot(xs, t, label=f"a={a_fixed}, s={s_}")
# plt.axvline(1.0, ls="--", alpha=0.5)
# plt.title("E[T] vs x (varying s, a fixed)")
# plt.xlabel("x (with terms=[x, -c], c=1)")
# plt.ylabel("E[T] (s)")
# plt.legend()
# plt.tight_layout()
# plt.show()


### Plotting loop

In [ ]:
# ========= Re-simulate with best params (LR-only, updated names) =========
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Pick which BO result to replay ---
best = res  # or whichever BO result dict you want

# --- Unpack ALL parameters returned by GP-BO ---
T_BEST    = 1.0 #float(best["T_enc"])
TOP_BEST  = float(best["T_op"])
A_BEST    = 1.2 #float(best["ddm_a"])     # DDM boundary
S_BEST    = 0.7 #float(best["ddm_s"])     # DDM noise
RT_BEST   = -2.0 #float(best["retrieval_threshold"])
LAT_BEST  = float(best["latency_factor"])
LAP_BEST  = 0.05 #float(best.get("lapse", 0.0))
COMPUTE_SF_BEST = int(best.get("compute_sf", 2))

print(
    f"Best params → T_enc={T_BEST:.4f}, T_op={TOP_BEST:.4f}, "
    f"RT={RT_BEST:.4f}, LAT={LAT_BEST:.4f}, ddm_a={A_BEST:.4f}, ddm_s={S_BEST:.4f}, "
    f"lapse={LAP_BEST:.4f}, compute_sf={COMPUTE_SF_BEST}"
)
print(
    f"Participant Id: {participant_id}, "
    f"Condition: {loader.get_participant_info(participant_id)['condition']}, "
    f"Complexity: {loader.get_participant_info(participant_id)['complexity']}"
)

# ---------------- Helpers ----------------
def _select_prob(probs, y):
    if hasattr(probs, "__len__") and len(probs) == 2:
        return float(probs[1]) if float(y) > 0 else float(probs[0])
    raise ValueError(f"Unexpected probs format: {probs}")

def _bool_with_xai(v):
    s = str(v).strip().lower()
    return s in ("w/ xai","with xai","xai","1","true","yes")

def _norm_label(with_xai_bool):
    return "w/ XAI" if with_xai_bool else "w/o XAI"

# ---------------- Strategy helpers ----------------
def _seed_memory_for_strategy(strategy, memory, explainer, compute_sf):
    if strategy == "lr_calc":
        add_lr_calculation_to_memory(explainer, memory)
    elif strategy == "lr_heur":
        add_lr_heuristic_to_memory(explainer, memory, initial_var=0.1)
    elif strategy == "dt":
        add_dt_to_memory(memory, explainer, thresh_sf=int(compute_sf))
    else:
        raise ValueError(f"Unknown strategy: {strategy!r}")

def _predict_once_verbose(strategy, instance, norm_instance, memory, explainer, *, with_xai, T_enc, T_op, ddm_a, ddm_s, compute_sf):
    if strategy == "lr_calc":
        return lr_calculation(
            instance, memory, lr_exp=explainer,
            T_enc=T_enc, T_op=T_op, ddm_a=ddm_a, ddm_s=ddm_s,
            compute_sf=compute_sf, mode=("read" if with_xai else "retrieve"), verbose=False
        )
    elif strategy == "lr_heur":
        # map T_enc -> T_READ_NUM, T_op -> T_INTUITIVE_OP
        return lr_heuristic(
            norm_instance, memory, explainer,
            num_samples=200, K_top=3,
            T_enc=T_enc,
            ddm_a=ddm_a, ddm_s=ddm_s, ddm_Tnd=0.30, ddm_norm="l2",
            active_indices=None, verbose=True
        )
    elif strategy == "dt":
        return dt_traverse(
            instance, memory, explainer,
            mode=("read" if with_xai else "retrieve"),
            compute_sf=int(compute_sf),
            T_enc=T_enc, ddm_a=ddm_a, ddm_s=ddm_s, ddm_Tnd=0.30, ddm_norm="l2",
            n_mc=64, topk_k=3, refresh_prob_cap=1.0, verbose=True
        )
    else:
        raise ValueError(f"Unknown strategy: {strategy!r}")

def _post_refresh(strategy, memory, explainer, *, compute_sf, info_or_aux, instance, with_xai, update_label):
    if strategy == "lr_calc":
        if with_xai:
            refresh_lr_calculation_in_memory(
                memory, explainer,
                intercept_display_sf=int(compute_sf),
                factor_display_sf=int(compute_sf)
            )
    elif strategy == "lr_heur":
        refresh_lr_heuristic_in_memory(
            memory, explainer, info_or_aux, actual=int(update_label),
            active_indices=None, w_min=1e-4, verbose=True
        )
    elif strategy == "dt":
        if with_xai:
            refresh_dt_path_in_memory(memory, explainer, instance, thresh_sf=int(compute_sf))

# ---------------- Strategy-aware forward simulation ----------------
def simulate_forward_df(
    T_enc, T_op, ddm_a, ddm_s, retrieval_threshold, latency_factor, compute_sf, lapse,
    forward_trials, ai_dataset_loader, explainer, *, strategy: str = "lr_calc"
):
    """
    Returns:
      df_best: DataFrame with columns:
        - trial
        - instance_id
        - with_xai
        - participant_response
        - prob_of_participant_response
        - model_time
        - participant_time
        - residual_time (= participant_time - model_time)
      memory: the final memory object after simulation
    """
    memory = _make_memory(retrieval_threshold, latency_factor)
    _seed_memory_for_strategy(strategy, memory, explainer, compute_sf)
    memory.tick(90)

    recs = []
    trials = forward_trials.sort_values("Trial Index")

    for _, row in trials.iterrows():
        trial_number  = int(row["Trial Index"])
        instance_id   = int(row["Instance Id"])
        actual_resp   = float(row["Response"])
        resp_time     = float(row["Time"])
        with_xai      = _bool_with_xai(row.get("Tested w/ XAI", "w/o XAI"))

        instances, preds = ai_dataset_loader.load_instances([instance_id], normalize=False)
        instance = instances[0]

        norm_instance = ai_dataset_loader.load_instances([instance_id], normalize=True)[0][0]

        probs, pred_time, info = _predict_once_verbose(
            strategy, instance, norm_instance, memory, explainer,
            with_xai=with_xai, T_enc=T_enc, T_op=T_op,
            ddm_a=ddm_a, ddm_s=ddm_s, compute_sf=compute_sf
        )

        # probability for the participant's response (with optional lapse)
        p_chosen = _select_prob(probs, actual_resp)
        if lapse > 0:
            p_chosen = (1.0 - lapse) * p_chosen + 0.5 * lapse

        # choose label for online update (heuristic updates both branches)
        if strategy == "lr_heur":
            if not with_xai:
                update_label = preds[0]
            else:
                update_label = int(explainer.apply_to_instance(instance) > 0)
        else:
            update_label = None
        _post_refresh(
            strategy, memory, explainer,
            compute_sf=compute_sf, info_or_aux=info,
            instance=instance, with_xai=with_xai, update_label=update_label
        )

        print(
            f"Trial {trial_number} [{_norm_label(with_xai)}] • Instance {instance_id} | "
            f"Participant response: {actual_resp:.0f} | "
            f"P(model on participant response): {p_chosen:.3f} | "
            f"Participant time: {resp_time:.3f}s | Model time: {pred_time:.3f}s"
        )
        print("=" * 80)

        model_pred = np.random.choice([0, 1], p=probs)

        recs.append({
            "trial": trial_number,
            "instance_id": instance_id,
            "with_xai": _norm_label(with_xai),
            "participant_response": actual_resp,
            "prob_of_participant_response": float(p_chosen),
            "model_time": float(pred_time),
            "participant_time": float(resp_time),
            "residual_time": float(resp_time - float(pred_time)),
            "participant_correct": int(int(actual_resp>0) == int(preds[0])),
            "model_correct": int(int(model_pred>0) == int(preds[0])),
        })

    df_best = pd.DataFrame.from_records(recs).sort_values("trial").reset_index(drop=True)
    return df_best, memory

# ---- Run the simulation with the fitted params (LR-only) ----
try:
    explainer.print_tree()
except Exception:
    pass

df_best, memory = simulate_forward_df(
    T_BEST, TOP_BEST, A_BEST, S_BEST, RT_BEST, LAT_BEST, COMPUTE_SF_BEST, LAP_BEST,
    forward_trials, ai_dataset_loader1, explainer, strategy=strategy
)


# --------- Split & quick summary ---------
def split_by_xai(df):
    df = df.copy().sort_values("trial")
    df["trial_seq"] = df.groupby("with_xai").cumcount() + 1
    return (
        df[df["with_xai"] == "w/ XAI"].reset_index(drop=True),
        df[df["with_xai"] == "w/o XAI"].reset_index(drop=True),
    )

df_wxai, df_woxai = split_by_xai(df_best)
print(f"\nCounts → w/ XAI: {len(df_wxai)} | w/o XAI: {len(df_woxai)}")
print("Participant Accuracy (w/ XAI):", (df_wxai["participant_correct"].mean() if len(df_wxai)>0 else float('nan')))
print("Participant Accuracy (w/o XAI):", (df_woxai["participant_correct"].mean() if len(df_woxai)>0 else float('nan')))
print("Model Accuracy (w/ XAI):", (df_wxai["model_correct"].mean() if len(df_wxai)>0 else float('nan')))
print("Model Accuracy (w/o XAI):", (df_woxai["model_correct"].mean() if len(df_woxai)>0 else float('nan')))

for label, sub in [("w/ XAI", df_wxai), ("w/o XAI", df_woxai)]:
    if len(sub) == 0:
        print(f"{label}: no trials")
        continue
    print(f"\n[{label}]")
    print("Mean residual (participant_time - model_time) [s]:", sub["residual_time"].mean())
    print("Median abs residual (s):", sub["residual_time"].abs().median())
    print("Avg P(model on participant response):", sub["prob_of_participant_response"].mean())

# --------- Plots ---------
# 1) Time series: model vs participant time per trial
plt.figure()
plt.plot(df_best["trial"], df_best["participant_time"], marker="o", label="Participant time")
plt.plot(df_best["trial"], df_best["model_time"], marker="s", label="Model time")
plt.xlabel("Trial"); plt.ylabel("Time (s)")
plt.title("Response Time per Trial • fitted params (LR-only)")
plt.legend(); plt.tight_layout(); plt.show()

# 2) Scatter: model_time vs participant_time
for label, sub in [("w/ XAI", df_wxai), ("w/o XAI", df_woxai)]:
    if len(sub) == 0:
        continue
    plt.figure()
    plt.scatter(sub["model_time"], sub["participant_time"], marker="o")
    dmin = float(min(sub["model_time"].min(), sub["participant_time"].min()))
    dmax = float(max(sub["model_time"].max(), sub["participant_time"].max()))
    plt.plot([dmin, dmax], [dmin, dmax])
    plt.xlabel("Model time (s)"); plt.ylabel("Participant time (s)")
    plt.title(f"Model vs Participant Time • {label} • fitted params")
    plt.tight_layout(); plt.show()

# 3) Probability assigned to the participant's chosen response
for label, sub in [("w/ XAI", df_wxai), ("w/o XAI", df_woxai)]:
    if len(sub) == 0:
        continue
    plt.figure()
    plt.plot(sub["trial_seq"], sub["prob_of_participant_response"], marker="o")
    plt.ylim(0.0, 1.0)
    plt.xlabel("Trial (within subset)")
    plt.ylabel("P(model assigns to participant's response)")
    plt.title(f"P(participant response) • {label} • fitted params")
    avg_nll = -np.log(sub["prob_of_participant_response"].clip(1e-12, 1.0)).mean()
    plt.figtext(0.5, -0.1, f"Average NLL = {avg_nll:.3f}", ha="center", fontsize=10)
    plt.tight_layout(); plt.show()

# 4) Time series by XAI condition (95th pct filter)
for label in ["w/ XAI", "w/o XAI"]:
    sub = df_best[df_best["with_xai"] == label]
    sub = sub[(sub["participant_time"] < sub["participant_time"].quantile(0.95))]
    if len(sub) == 0:
        continue
    plt.figure()
    plt.plot(sub["trial"], sub["participant_time"], marker="o", label="Participant time")
    plt.plot(sub["trial"], sub["model_time"], marker="s", label="Model time")
    plt.xlabel("Trial"); plt.ylabel("Time (s)")
    plt.title(f"Response Time per Trial • {label} • fitted params")
    plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# get 40 random instances
# instances = random.sample(range(400), 40)
# correct = 0

# for instance_id in instances:
#     instances, preds = ai_dataset_loader1.load_instances([instance_id], normalize=False)
#     instance = instances[0]
#     preds = preds[0]
#     print(f"Instance {instance_id}: {instance}")

#     p, t, _ = dt_traverse(
#         instance, memory, explainer,
#         mode="retrieve",
#     )

#     model_pred = np.random.choice([0, 1], p=p)

#     correct += int(model_pred == preds)

# print(f"Accuracy on 40 random instances: {correct/40:.3f}")
    

In [ ]:
memory.dm.time


In [ ]:
def round_sig(x, k):
    if k is None or x == 0: 
        return x
    p = math.floor(math.log10(abs(x)))
    f = 10**(p - k + 1)
    return round(x / f + 1e-12) * f   # small epsilon fixes borderline like 10.5

print(round_sig(10.5, 2))  # -> 10.0 (with your version)


## Fitting multiple participants

In [ ]:
import random
import pandas as pd

# --- config reused from your snippet ---
# Config
cfg = ObjConfig(
    w_resp=1.0, w_time=0.2, repeats=1,
    T_enc_bounds=(0.5, 3.0),
    T_op_bounds=(2.0, 3.0),
    retrieval_threshold_bounds=(-2.0, 1.5),
    latency_factor_bounds=(0.001, 1.0),
    ddm_a_bounds=(1.0, 1.5),
    ddm_s_bounds=(0.2, 1.5),
    lapse_bounds=(0.05, 0.2),
    compute_sf_bounds=(1.0, 3.0)
)

init_vals = {
        "T_enc": 1.5, "T_op": 2.0, "retrieval_threshold": -0.5,
        "latency_factor": 0.3, "ddm_a": 1.0, "ddm_s": 1.0,
        "lapse": 0.1, "compute_sf": 2.0,
}


# random.seed(42)

# 1) pick k LR participants (without replacement)
all_pids = loader.get_participant_ids()
lr_pids = [pid for pid in all_pids if loader.get_participant_info(pid)['condition'] == 'LR']
picked_pids = random.sample(lr_pids, 45)

rows = []

for participant_id in picked_pids:
    print(f"\n=== Optimizing for participant {participant_id} (LR) ===")
    info = loader.get_participant_info(participant_id)
    app_id     = info['app_id']
    model_name = info['model']
    complexity = info['complexity']
    condition  = info['condition']  # 'LR'

    # Explainer for LR condition
    variant = "sparse" if complexity == 'low' else "dense"
    explainer = LogisticRegressionInterpreter(lr_df, metadata_df, app_id, model_name, variant=variant)
    model_type = "lr"

    # Filter dataset loader to this app/model
    ai_dataset_loader1 = filter_by_app_and_model(ai_dataset_loader, app_id, model_name)

    # Trials
    forward_trials = loader.get_forward_trials(participant_id)

    # 2) Fit with CALCULATION model
    res_calc = fit_params_gp_bo(
        forward_trials, ai_dataset_loader1, explainer, model_name, app_id=app_id, strategy="lr_calc",
        init_vals=init_vals,
        n_calls=50, n_initial_points=1, random_state=42,
        cfg=cfg, verbose=True,
        freeze={
            "T_op": 2.0,            # freeze T_op to 2.0
            "lapse": 0.05,          # freeze lapse
            "compute_sf": 2,        # freeze compute_sf
            "latency_factor": 1.0,
            # "ddm_a": 1.0,
            # "T_enc": 1.5
        }
    )

    # 3) Fit with HEURISTIC model
    res_heur = fit_params_gp_bo(
        forward_trials, ai_dataset_loader1, explainer, model_name, app_id=app_id, strategy="lr_heur",
        init_vals=init_vals,
        n_calls=50, n_initial_points=1, random_state=42,
        cfg=cfg, verbose=True,
        freeze={
            "T_op": 2.0,            # freeze T_op to 2.0
            "lapse": 0.05,          # freeze lapse
            "compute_sf": 2,        # freeze compute_sf
            "latency_factor": 1.0,
            # "ddm_a": 1.0,
            # "T_enc": 1.5
        }
    )

    obj_calc = res_calc.get("objective")
    obj_heur = res_heur.get("objective")

    # print("Best (calculation):", {k: v for k, v in res_calc.items() if k in ["T_READ_NUM","retrieval_threshold","latency_factor","W0_ANS","lapse", "compute_sf", "objective"]})
    # print("Best (heuristic):   ", {k: v for k, v in res_heur.items() if k in ["T_READ_NUM","retrieval_threshold","latency_factor","W0_ANS","lapse", "compute_sf", "objective"]})

    row = {
        "Participant Id": participant_id,
        "AppId": app_id,
        "Model": model_name,
        "Complexity": complexity,
        "Variant": variant,
    }

    for strategy in ["calculation", "heuristic"]:
        res = res_calc if strategy == "calculation" else res_heur
        new_row = row.copy()
        new_row.update({
            "Strategy": strategy,
            # everything in res except history
            **{k: (v if not isinstance(v, float) else f"{v:.3f}") for k, v in res.items() if k != "history"},
        })
        rows.append(new_row)
        # print(new_row)


    # rows.append({
    #     "Participant Id": participant_id,
    #     "AppId": app_id,
    #     "Model": model_name,
    #     "Complexity": complexity,
    #     "Variant": variant,
    #     "Objective Calculation": obj_calc,
    #     "Objective Heuristic": obj_heur,
    #     "better_model": "calculation" if obj_calc < obj_heur else ("heuristic" if obj_heur < obj_calc else "tie"),
    #     "res_calc": res_calc,
    #     "res_heur": res_heur,
    # })


# 4) Summary table of objectives
priority_cols = ["Participant Id", "AppId", "Model", "Complexity", "Variant"]
all_cols = priority_cols + [c for c in pd.DataFrame(rows).columns if c not in priority_cols]
summary_df = pd.DataFrame(rows)[all_cols]

print("\n=== Objective summary (lower is better) ===")
print(summary_df.to_string(index=False))

summary_df.to_csv("participant_parameters_fit_lr_v0.1.csv", index=False)

In [ ]:
import random
import pandas as pd

# --- config reused from your snippet ---
# Config
cfg = ObjConfig(
    w_resp=1.0, w_time=0.2, repeats=1,
    T_enc_bounds=(0.5, 3.0),
    T_op_bounds=(2.0, 3.0),
    retrieval_threshold_bounds=(-2.0, 1.5),
    latency_factor_bounds=(0.001, 1.0),
    ddm_a_bounds=(1.0, 1.5),
    ddm_s_bounds=(0.7, 1.5),
    lapse_bounds=(0.05, 0.2),
    compute_sf_bounds=(1.0, 3.0)
)

init_vals = {
        "T_enc": 1.5, "T_op": 2.0, "retrieval_threshold": -0.5,
        "latency_factor": 0.3, "ddm_a": 1.0, "ddm_s": 1.0,
        "lapse": 0.1, "compute_sf": 2.0,
}


# random.seed(42)

# 1) pick k DT participants (without replacement)
all_pids = loader.get_participant_ids()
dt_pids = [pid for pid in all_pids if loader.get_participant_info(pid)['condition'] == 'DT']
picked_pids = random.sample(dt_pids, 45)

rows = []

for participant_id in picked_pids:
    print(f"\n=== Optimizing for participant {participant_id} (DT) ===")
    info = loader.get_participant_info(participant_id)
    app_id     = info['app_id']
    model_name = info['model']
    complexity = info['complexity']
    condition  = info['condition']  # 'DT'

    # Explainer for DT condition
    explainer = DecisionTreeInterpreter(dt_df, metadata_df, app_id, model_name, depth=3 if complexity == 'high' else 2)
    model_type = "dt"

    # Filter dataset loader to this app/model
    ai_dataset_loader1 = filter_by_app_and_model(ai_dataset_loader, app_id, model_name)

    # Trials
    forward_trials = loader.get_forward_trials(participant_id)

    # 2) Fit with CALCULATION model
    res = fit_params_gp_bo(
        forward_trials, ai_dataset_loader1, explainer, model_name, app_id=app_id, strategy="dt",
        init_vals=init_vals,
        n_calls=50, n_initial_points=1, random_state=42,
        cfg=cfg, verbose=True,
        freeze={
            "T_op": 2.0,            # freeze T_op to 2.0
            "lapse": 0.05,          # freeze lapse
            "compute_sf": 2,        # freeze compute_sf
            "latency_factor": 1.0,
            # "ddm_a": 1.0,
            # "T_enc": 1.5
        }
    )

    # 3) Fit with HEURISTIC model
    # res_heur = fit_params_gp_bo(
    #     forward_trials, ai_dataset_loader1, explainer, model_name, app_id=app_id, strategy="lr_heur",
    #     init_vals=init_vals,
    #     n_calls=50, n_initial_points=1, random_state=42,
    #     cfg=cfg, verbose=True,
    #     freeze={
    #         "T_op": 2.0,            # freeze T_op to 2.0
    #         "lapse": 0.05,          # freeze lapse
    #         "compute_sf": 2,        # freeze compute_sf
    #         "latency_factor": 1.0,
    #         # "ddm_a": 1.0,
    #         # "T_enc": 1.5
    #     }
    # )

    # obj_calc = res_calc.get("objective")
    # obj_heur = res_heur.get("objective")

    # print("Best (calculation):", {k: v for k, v in res_calc.items() if k in ["T_READ_NUM","retrieval_threshold","latency_factor","W0_ANS","lapse", "compute_sf", "objective"]})
    # print("Best (heuristic):   ", {k: v for k, v in res_heur.items() if k in ["T_READ_NUM","retrieval_threshold","latency_factor","W0_ANS","lapse", "compute_sf", "objective"]})

    row = {
        "Participant Id": participant_id,
        "AppId": app_id,
        "Model": model_name,
        "Complexity": complexity,
        "Variant": variant,
    }
    new_row = row.copy()
    new_row.update({
        # "Strategy": strategy,
        # everything in res except history
        **{k: (v if not isinstance(v, float) else f"{v:.3f}") for k, v in res.items() if k != "history"},
    })
    rows.append(new_row)
    # print(new_row)


    # rows.append({
    #     "Participant Id": participant_id,
    #     "AppId": app_id,
    #     "Model": model_name,
    #     "Complexity": complexity,
    #     "Variant": variant,
    #     "Objective Calculation": obj_calc,
    #     "Objective Heuristic": obj_heur,
    #     "better_model": "calculation" if obj_calc < obj_heur else ("heuristic" if obj_heur < obj_calc else "tie"),
    #     "res_calc": res_calc,
    #     "res_heur": res_heur,
    # })


# 4) Summary table of objectives
priority_cols = ["Participant Id", "AppId", "Model", "Complexity", "Variant"]
all_cols = priority_cols + [c for c in pd.DataFrame(rows).columns if c not in priority_cols]
summary_df = pd.DataFrame(rows)[all_cols]

print("\n=== Objective summary (lower is better) ===")
print(summary_df.to_string(index=False))

summary_df.to_csv("participant_parameters_fit_dt_v0.1.csv", index=False)